# 00 - Configuration and data inventory

**Purpose**: find where the input data are (locally and on BDAP), check what they contain
(grid, period, calendar, units, CRS) and write a configuration file that all the other notebooks read.

This notebook corrects nothing. It only reads and reports problems.

**Do this once**: replace every `TO_FILL` (file patterns below), first locally, then after exploring BDAP
(STAC catalogue and `/eos/jeodpp/data`). Run it again after each change.

**This is the only place where data paths are entered.** The notebooks of `1_SVM/` read `config/config_<profile>.json`
written here (through `crop_utils.load_config()`). Data that are not in a BDAP catalogue (MIRCA2000, MIRCA-OS, GAEZ v4
`SQ*_mze`, `cell_area_ha.npy`, GAEZ appendices) are read directly by the notebooks from `0_Input_data/`. This folder is
not in git: on BDAP, create it in the cloned project and link its sub-folders to EOS (the GAEZ v4 appendix table
read by 2.c is not in git either):
```bash
CS=/eos/jeodpp/data/projects/DROUGHTOBS/elisa/Crop_suitability
cd ~/crop_suitability
mkdir -p 0_Input_data/masks_crops 1_SVM/2_soil_suitability_gaez/GAEZ_soil_ref/appendix_tables
ln -s $CS/MIRCA_2000 0_Input_data/masks_crops/MIRCA_2000
ln -s $CS/gaezv5/gaezv5_soil_suitability 0_Input_data/gaez
ln -s $CS/gaezv5/gaezv5_soil_suitability/GAEZv4_Appendices.xlsx 1_SVM/2_soil_suitability_gaez/GAEZ_soil_ref/appendix_tables/GAEZ4_Appendices.xlsx
```
(the notebooks find the project root as the first parent folder that contains `0_Input_data`).

HWSD v2 on BDAP (raster downloaded from FAO, attribute database uploaded as `HWSD2.sqlite`):
```bash
mkdir -p $CS/HWSD2 && cd $CS/HWSD2
curl -O https://s3.eu-west-1.amazonaws.com/data.gaezdev.aws.fao.org/HWSD/HWSD2_RASTER.zip
python -m zipfile -e HWSD2_RASTER.zip . && rm HWSD2_RASTER.zip
```
Results are written inside the project tree, next to each notebook.

Pipeline (`1_SVM/`):
0. `0_preprocessing_masks` - land mask, MIRCA2000 crop masks and ERA5 regridded to the 1 deg SVM grid;
1. `1_preprocessing_BIOvariables` - bioclimatic variables from ERA5 (1981-2015) and ISIMIP3b MRI-ESM2-0 SSP5-8.5 (2061-2080);
2. `2_soil_suitability`, `2_soil_suitability_gaez` - soil (HWSD v2 x GAEZ Module IV) and terrain-slope suitability;
3. `3_SVM_train` - SVM training (one branch per MIRCA2000 presence threshold);
4. `4_SVM_predict` - climate suitability, ERA5 and ISIMIP;
5. `5_Suitability_climate_soil_combined_validation` - climate x soil, validation against MIRCA2000 / MIRCA-OS / GAEZ v5;
6. `6_Suitability_climate_soil_combined_AMOC` - soil constraint on the AMOC scenarios;
7. `7_test_adaptable_calendar` - shifted / adapted sowing calendars.

**Running on BDAP**: use JEO-lab (JupyterLab, Python 3) with the conda environment of `environment.yml`.
The code uses only `pathlib` and forward slashes, so it runs unchanged on Linux and on Windows.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys
import glob
import json
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import rasterio

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------------------------------------------------------
# 1. Where is this notebook running? (BDAP = Linux with the EOS storage mounted)
# ----------------------------------------------------------------------------
ON_BDAP = Path("/eos/jeodpp").exists()
PROFILE = "bdap" if ON_BDAP else "local"
print("Active profile:", PROFILE)

# Project root = first parent folder containing 0_Input_data (this notebook sits in it)
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())

# ----------------------------------------------------------------------------
# 2. Data roots ({DATA} in the patterns below). Always use "/" in paths, never a backslash.
#    local: the PhD folder (data of the original crops_suitability_codes project, bias-adjustment outputs)
#    bdap : the BDAP data storage (the "bdap_data" folder seen in JupyterLab)
# ----------------------------------------------------------------------------
ROOTS = {
    "local": {"data": "C:/Users/elisa/OneDrive - Politecnico di Milano/PHD_codes"},
    "bdap":  {"data": "/eos/jeodpp/data"},
}
DATA_ROOT = ROOTS[PROFILE]["data"]
CONFIG_DIR = BASE / "config"
CONFIG_PATH = CONFIG_DIR / f"config_{PROFILE}.json"
print("Project root:", BASE)
print("Data root   :", DATA_ROOT)

## Python environment check

Lists the packages needed by the pipeline. On BDAP, create the environment once from `environment.yml`
(instructions at the top of that file), or install any missing package in your conda environment.

In [ ]:
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
import crop_utils as cu          # crop_utils.py sits next to this notebook
missing = cu.check_environment()
print("\nPlatform:", platform.system())

## The datasets

For each dataset, `glob_<profile>` is a file pattern. `{DATA}` is replaced by the data root and `{var}` by the
variable name (datasets without variables, e.g. rasters, have `vars = []`).
Example patterns: `{DATA}/base/Soil/GLOBAL/HWSD/LATEST/Data/GeoTIFF/*.tif` or `{DATA}/<...>/era5/{var}/*.nc`.

You can use a different pattern locally and on BDAP (`glob_local`, `glob_bdap`). A variable can have its own pattern
(e.g. `glob_bdap_pr`), and a pattern can be a **list**: the files of all the patterns are combined.

`kind` tells the inventory how to read the files: `netcdf` (xarray), `raster` (rasterio) or `table` (HWSD2 Access database).
`used_by` lists the notebooks that read the dataset. `separate_runs` marks datasets with one file per scenario
(not one file per time chunk).

As long as a pattern contains `TO_FILL`, the inventory skips it and says so. A dataset with `skip` (the reason) is not
needed for the current work: the inventory does not look for it.

**HWSD v2 (soil)**: the notebooks need **two** things: the raster of soil mapping units (`HWSD2_SMU_ID` per pixel,
~1 km, a GeoTIFF is fine) **and** the attribute database `HWSD2.mdb` (tables `HWSD2_SMU`, `HWSD2_LAYERS`, `D_*`:
texture, drainage, depth, phases... per soil component). A GeoTIFF of mapping units does not contain the attributes.
The `.mdb` is read with `pyodbc` + the Microsoft Access driver, which exist only on Windows, so BDAP uses a SQLite
copy of it (`HWSD2.sqlite`, same tables, written on Windows by `python hwsd2_mdb_to_sqlite.py HWSD2.mdb`); the
notebooks open either through `crop_utils.hwsd_connect()`. BDAP `base/Soil/GLOBAL/HWSD/LATEST` is HWSD **v1.2**
(`HWSD.mdb`, `hwsd.tif`: other tables and mapping units), so HWSD v2 is put in the project folder on EOS: the raster
from FAO (`HWSD2_RASTER.zip`, see 'Running on BDAP' below) and the uploaded `HWSD2.sqlite`.

In [ ]:
# Possible variable names in the files (ERA5, AgERA5, ISIMIP and the bias-adjusted files do not use the same ones)
ALIASES = {
    "tasmax": ["tasmax", "mx2t", "tmax", "t2m_max", "Temperature_Air_2m_Max_24h"],
    "tasmin": ["tasmin", "mn2t", "tmin", "t2m_min", "Temperature_Air_2m_Min_24h"],
    "pr":     ["pr", "tp", "precip", "precipitation", "Precipitation_Flux"],
    "dswrf":  ["dswrf", "ssrd", "rsds", "msdwswrf", "Solar_Radiation_Flux"],
    "ssrd":   ["rsds", "ssrd", "dswrf", "msdwswrf"],
    "et0":    ["ET0", "et0", "ReferenceET", "Reference_evapotranspiration", "pet"],
}

A = "TO_FILL"
OLD = "{DATA}/crops_suitability_codes"   # local copy of the original project (inputs removed from this one)
GCM_IN = "{DATA}/projects/DROUGHTOBS/elisa/GCM_downscaling/Input_data"   # BDAP inputs of the GCM bias adjustment project
CS = "{DATA}/projects/DROUGHTOBS/elisa/Crop_suitability"                   # BDAP: inputs uploaded for this project
HOSING = "{DATA}/projects/DROUGHTOBS/EC-Earth_hosing"                    # BDAP: raw EC-Earth3 hosing runs
EC_LOCAL = "{DATA}/bias adjustment_codes/0)Input_data/EC-Earth3/historical_raw"   # local: raw EC-Earth3 historical (yearly files)


def gcm_run(period, run):
    """Raw EC-Earth3 stabilisation run on BDAP (same files as the GCM bias adjustment, read raw): tasmax / tasmin daily
    global in GLOBAL/<period>_<run>/; the pr files there are monthly, so daily pr comes from the northern (NH) and
    southern (SH) hemisphere files, to be glued together when read. rsds: location still to be found."""
    sh = f"stabilization-ssp585-{period}" + ("_Hosing" if run == "Hosing" else "")
    return {
        "kind": "netcdf", "vars": ["tasmax", "tasmin", "pr", "ssrd"], "used_by": "delta method (to come)",
        "glob_local": f"{{DATA}}/{A}/gcm_{period}_{run.lower()}/{{var}}/*.nc",
        "glob_bdap": f"{HOSING}/GLOBAL/{period}_{run}/{{var}}_*GLOBAL_??????-??????.nc",
        "glob_bdap_pr": [f"{HOSING}/stabilization-ssp585-{period}_{run}/pr_day_*_NH_??????-??????.nc",
                         f"{HOSING}/day_SH/{sh}/pr_day_*_SH_??????-??????.nc"],
        "glob_bdap_ssrd": f"{A}: rsds of the hosing runs (location to be found)",
    }


DATASETS = {
    # ---- climate, historical ---------------------------------------------------
    "era5": {   # local: daily, 0.25 deg, 1981-2015, one file per variable
                # BDAP: daily, regridded onto the EC-Earth3 grid (0.7 deg, CDO conservative, merged files, GCM_downscaling
                # project); radiation is in the rsds files
        "kind": "netcdf", "vars": ["tasmax", "tasmin", "pr", "dswrf"], "used_by": "0.1, 1.a, 2.f",
        "glob_local": f"{OLD}/0_Input_data/ERA5 data/{{var}}_ERA5_1981_2015_daily_025deg.nc",
        "glob_bdap":  f"{GCM_IN}/ERA5_daily_on_ecearth3grid/{{var}}_ERA5_day_*.nc",
        "glob_bdap_dswrf": f"{GCM_IN}/ERA5_daily_on_ecearth3grid/rsds_ERA5_day_*.nc",
    },
    "agera5_et0": {   # AgERA5 FAO-56 Penman-Monteith reference ETo, daily files (ETo formula check only)
                      # BDAP: one file per day in consolidated/<yyyy>/<mm>/<dd>/
        "kind": "netcdf", "vars": ["et0"], "used_by": "1.a",
        "glob_local": f"{OLD}/0_Input_data/ERA5 data/AgERA5/Reference evapotranspiration/*.nc",
        "glob_bdap":  ("{DATA}/base/Meteo/GLOBAL/ECMWF/Reanalysis/AgERA5/VER2-0/Data/NetCDF/reference_evapotranspiration/"
                       "consolidated/*/*/*/ReferenceET-PenmanMonteith-FAO56_C3S-glob-agric_AgERA5_*_final-v2.0.0.nc"),
    },
    # ---- climate, future -------------------------------------------------------
    "isimip_mri_ssp585": {   # ISIMIP3b MRI-ESM2-0 SSP5-8.5 daily, 0.5 deg; one folder per variable (rsds in ssrd/)
        "kind": "netcdf", "vars": ["tasmax", "tasmin", "pr", "ssrd"], "used_by": "1.2.a, 7.b, 7.e",
        "skip": "future scenario of the earlier study, replaced by the EC-Earth3 runs (BDAP CMIP6 MRI-ESM2-0 has no rsds)",
        "glob_local": f"{OLD}/0_Input_data/ISIMIP_MRI-ESM2-0/ssp585/{{var}}/mri-esm2-0_r1i1p1f1_w5e5_ssp585_*_global_daily_*.nc",
        "glob_bdap":  f"{{DATA}}/{A}/ISIMIP3b/MRI-ESM2-0/ssp585/{{var}}/*.nc",
    },
    # ---- climate, EC-Earth3 raw (AMOC scenarios, delta method on the BIO variables) ----
    "gcm_historical": {   # EC-Earth3 historical CMIP6 r1i1p1f1, daily, global, 0.7 deg (merged with CDO on BDAP)
        "kind": "netcdf", "vars": ["tasmax", "tasmin", "pr", "ssrd"], "used_by": "delta method (to come)",
        "glob_local": f"{EC_LOCAL}/{{var}}_day_EC-Earth3_historical_r1i1p1f1_gr_*.nc",
        "glob_local_ssrd": f"{EC_LOCAL}/rsds_day_EC-Earth3_historical_r1i1p1f1_gr_*.nc",
        "glob_bdap": f"{GCM_IN}/EC-Earth3_historical_CMIP6_merged/{{var}}_day_EC-Earth3_historical_r1i1p1f1_*.nc",
        "glob_bdap_ssrd": f"{GCM_IN}/EC-Earth3_historical_CMIP6/rsds_day_EC-Earth3_historical_r1i1p1f1_*.nc",   # yearly files
    },
    "gcm_2025_nohosing": gcm_run("2025", "Original"),
    "gcm_2025_hosing": gcm_run("2025", "Hosing"),
    "gcm_2050_nohosing": gcm_run("2050", "Original"),
    "gcm_2050_hosing": gcm_run("2050", "Hosing"),
    # ---- soil and terrain ------------------------------------------------------
    "hwsd_raster": {   # HWSD v2 soil mapping units (SMU id per pixel); BDAP base/Soil has v1.2 only -> uploaded
        "kind": "raster", "vars": [], "used_by": "2.a, 2.c, 2.d, 2.e, 5.a-5.d",
        "glob_local": f"{OLD}/0_Input_data/soil/HWSD2.bil",
        "glob_bdap":  f"{CS}/HWSD2/HWSD2.bil",
    },
    "hwsd_mdb": {   # HWSD v2 attribute database (BDAP: SQLite copy of the .mdb, see above)
        "kind": "table", "vars": [], "used_by": "2.a, 2.c",
        "glob_local": f"{OLD}/0_Input_data/soil/HWSD2.mdb",
        "glob_bdap":  f"{CS}/HWSD2/HWSD2.sqlite",
    },
    "dem_slope": {   # optional slope raster for 2.a (the slope step is skipped if absent)
        "kind": "raster", "vars": [], "used_by": "2.a (optional)", "skip": "terrain suitability not done for now",
        "glob_local": f"{OLD}/0_Input_data/soil/DEM_slope.tif",
        "glob_bdap":  f"{{DATA}}/{A}/DEM_slope.tif",
    },
    "gaez_slope": {   # GAEZ terrain median slope
        "kind": "raster", "vars": [], "used_by": "2.f", "skip": "terrain suitability not done for now",
        "glob_local": f"{OLD}/0_Input_data/terrain/SLOPE-MED.tif",
        "glob_bdap":  f"{{DATA}}/{A}/SLOPE-MED.tif",
    },
    # ---- GAEZ v5 reference suitability (validation) ----------------------------
    "gaez_v5_hist": {
        "kind": "raster", "vars": [], "used_by": "5.a-5.f",
        "glob_local": f"{OLD}/1_check_GAEZ/gaez_v5_hp8100/GAEZ-V5.RES05-SIX.HP8100.AGERA5.HIST.*.tif",
        "glob_bdap":  f"{CS}/gaezv5/gaezv5_crop_suitability/gaez_crop_suit_hp8100/GAEZ-V5.RES05-SIX.HP8100.AGERA5.HIST.*.tif",
    },
    "gaez_v5_future": {
        "kind": "raster", "vars": [], "used_by": "5.g.bis",
        "glob_local": f"{OLD}/1_check_GAEZ/gaez_v5_future_mri_esm2_ssp585/GAEZ-V5.RES05-SXX.FP6180.MRI-ESM2-0.SSP585.*.tif",
        "glob_bdap":  f"{CS}/gaezv5/gaezv5_crop_suitability/gaez_crop_suit_MRI_ESM2/GAEZ-V5.RES05-SXX.FP6180.MRI-ESM2-0.SSP585.*.tif",
    },
}

## Inventory functions

NetCDF datasets: only the first and the last file are opened, lazily (nothing is loaded into memory).
Rasters: only the metadata of the first file are read.

In [ ]:
try:                                   # recent xarray: CFDatetimeCoder ; otherwise use_cftime
    TIME_KW = {"decode_times": xr.coders.CFDatetimeCoder(use_cftime=True)}
except AttributeError:
    TIME_KW = {"use_cftime": True}


def var_keys(spec):
    return spec["vars"] or ["file"]


def resolve_pattern(spec, var, template=False):
    """File pattern(s) for the active profile, with {DATA} (and {var}, unless template=True) replaced.
    `glob_<profile>_<var>` overrides `glob_<profile>`; a list of patterns is returned as a list."""
    pattern = spec[f"glob_{PROFILE}"] if template else spec.get(f"glob_{PROFILE}_{var}", spec[f"glob_{PROFILE}"])
    def fill(p):
        p = p.replace("{DATA}", str(DATA_ROOT))
        return p if template else p.replace("{var}", var)
    return [fill(p) for p in pattern] if isinstance(pattern, list) else fill(pattern)


def as_list(pattern):
    return pattern if isinstance(pattern, list) else [pattern]


def find_var(ds, var):
    """Find the real variable name in the file from the aliases (or the only data variable)."""
    for name in ALIASES.get(var, [var]):
        if name in ds.data_vars:
            return name
    data_vars = [v for v in ds.data_vars if ds[v].ndim >= 2]
    return data_vars[0] if len(data_vars) == 1 else None


def find_coord(ds, options):
    for name in options:
        if name in ds.coords or name in ds.dims:
            return name
    return None


def unit_hint(var, units):
    """Conversion needed to obtain mm/day, degrees Celsius or W m-2."""
    u = (units or "").strip().lower().replace(" ", "")
    if var in ("pr", "et0"):
        if u in ("kgm-2s-1", "kgm**-2s**-1", "kg/m2/s", "mm/s", "kgm^-2s^-1"):
            return "x 86400 -> mm/day"
        if u in ("m", "m/day", "md-1"):
            return "x 1000 -> mm/day"
        if u in ("mm/day", "mmday-1", "mmd-1", "mm", "mmd**-1"):
            return "ok (mm/day)"
        return "check units"
    if var in ("tasmax", "tasmin"):
        if u in ("k", "kelvin"):
            return "- 273.15 -> degC"
        if u in ("degc", "c", "celsius", "degreec", "degrees_c", "°c"):
            return "ok (degC)"
        return "check units"
    if var in ("dswrf", "ssrd"):
        if u in ("wm-2", "w/m2", "wm**-2", "w/m^2", "wm^-2"):
            return "ok (W m-2)"
        if u in ("jm-2", "j/m2", "jm**-2", "jm^-2"):
            return "J m-2 (daily sum): / 86400 -> W m-2"
        return "check units"
    return ""


def describe_grid(ds, row, flags):
    latn = find_coord(ds, ["lat", "latitude", "y"])
    lonn = find_coord(ds, ["lon", "longitude", "x"])
    if latn and lonn and ds[latn].ndim == 1 and ds[lonn].ndim == 1:
        lat, lon = ds[latn].values, ds[lonn].values
        row["n_lat"], row["n_lon"] = lat.size, lon.size
        row["res_lat"] = round(float(np.median(np.abs(np.diff(lat)))), 4) if lat.size > 1 else None
        row["res_lon"] = round(float(np.median(np.abs(np.diff(lon)))), 4) if lon.size > 1 else None
        row["lat_range"] = f"{lat.min():.2f}..{lat.max():.2f}" + (" (descending)" if lat[0] > lat[-1] else "")
        row["lon_range"] = f"{lon.min():.2f}..{lon.max():.2f}"
        if lon.max() > 180:
            flags.append("lon 0-360")
    else:
        flags.append("irregular grid or unknown lat/lon names")


def inventory_netcdf(row, files, var, separate_runs=False):
    """separate_runs=True: one file per scenario (not time chunks) -> period of the first file only."""
    flags = []
    try:
        first = xr.open_dataset(files[0], **TIME_KW)
        last = xr.open_dataset(files[-1], **TIME_KW) if len(files) > 1 and not separate_runs else first
    except Exception as exc:
        row["status"] = f"error while opening: {exc}"
        return row
    vname = find_var(first, var)
    if vname is None:
        row["status"] = f"variable missing (looked for: {ALIASES.get(var, [var])}; present: {list(first.data_vars)})"
        return row
    da = first[vname]
    row["name_in_file"] = vname
    row["units"] = da.attrs.get("units", "?")
    row["unit_hint"] = unit_hint(var, da.attrs.get("units"))
    if "check" in row["unit_hint"]:
        flags.append("units")
    describe_grid(first, row, flags)
    if "time" in first.coords:
        t0, t1 = first["time"].values, last["time"].values
        row["t_start"], row["t_end"] = str(t0[0])[:10], str(t1[-1])[:10]
        try:
            row["calendar"] = first["time"].dt.calendar
        except Exception:
            row["calendar"] = "?"
        if t0.size > 1:
            steps = np.array([(b - a).total_seconds() / 86400 for a, b in zip(t0[:-1], t0[1:])])
            row["step_days"] = round(float(np.median(steps)), 4)
            if abs(row["step_days"] - 1.0) > 1e-6:
                flags.append("time step is not one day")
        row["n_time_first_file"] = int(t0.size)
    else:
        flags.append("no time coordinate")
    if separate_runs:
        row["files"] = ", ".join(Path(f).name for f in files[:6]) + (" ..." if len(files) > 6 else "")
        row["t_start"] = f"{row.get('t_start')} (1st file)"
    first.close(); last.close()
    row["status"] = "ok" if not flags else "ok, check: " + "; ".join(flags)
    return row


def inventory_raster(row, files):
    flags = []
    try:
        with rasterio.open(files[0]) as src:
            row["crs"] = src.crs.to_string() if src.crs else "none"
            row["res_lat"], row["res_lon"] = round(abs(src.res[1]), 6), round(abs(src.res[0]), 6)
            row["n_lat"], row["n_lon"] = src.height, src.width
            b = src.bounds
            row["lat_range"] = f"{b.bottom:.2f}..{b.top:.2f}"
            row["lon_range"] = f"{b.left:.2f}..{b.right:.2f}"
            row["dtype"], row["nodata"], row["bands"] = src.dtypes[0], src.nodata, src.count
            if src.crs is None or not src.crs.is_geographic:
                flags.append(f"CRS {row['crs']} (geographic lon/lat expected)")
    except Exception as exc:
        row["status"] = f"error while opening: {exc}"
        return row
    if len(files) > 1:
        row["files"] = ", ".join(Path(f).name for f in files[:6]) + (" ..." if len(files) > 6 else "")
    row["status"] = "ok" if not flags else "ok, check: " + "; ".join(flags)
    return row


def inventory_table(row, files):
    if platform.system() != "Windows" and not files[0].lower().endswith(".sqlite"):
        row["status"] = "ok, check: Access .mdb readable with pyodbc only on Windows (2.a / 2.c)"
    else:
        row["status"] = "ok"
    return row


def inventory_one(name, spec, var):
    pattern = resolve_pattern(spec, var)
    row = {"dataset": name, "var": "" if var == "file" else var, "kind": spec["kind"], "used_by": spec["used_by"],
           "n_files": 0, "status": "", "pattern": " + ".join(as_list(pattern))}
    if spec.get("skip"):
        row["status"] = f"skipped: {spec['skip']}"
        return row
    if any("TO_FILL" in p for p in as_list(pattern)):
        row["status"] = "to fill in"
        return row
    files = sorted(f for p in as_list(pattern) for f in glob.glob(p))
    row["n_files"] = len(files)
    if not files:
        row["status"] = "no file found"
        return row
    row["size_gb"] = round(sum(os.path.getsize(f) for f in files) / 1e9, 3)
    if spec["kind"] == "netcdf":
        return inventory_netcdf(row, files, var, spec.get("separate_runs", False))
    if spec["kind"] == "raster":
        return inventory_raster(row, files)
    return inventory_table(row, files)

## Run the inventory

The resulting table is saved as `config/inventory_<profile>.csv`. Rows saying "to fill in" are normal
as long as the paths are not filled in; "skipped" rows are not needed for the current work.

In [ ]:
rows = []
for name, spec in DATASETS.items():
    for var in var_keys(spec):
        rows.append(inventory_one(name, spec, var))

inventory = pd.DataFrame(rows)
cols = ["dataset", "var", "kind", "status", "n_files", "name_in_file", "units", "unit_hint",
        "res_lat", "res_lon", "lat_range", "lon_range", "t_start", "t_end", "calendar", "step_days",
        "crs", "dtype", "nodata", "bands", "size_gb", "used_by", "files", "pattern"]
inventory = inventory.reindex(columns=cols)

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
inventory.to_csv(CONFIG_DIR / f"inventory_{PROFILE}.csv", index=False)
with pd.option_context("display.max_columns", None, "display.width", 250, "display.max_colwidth", 60):
    display(inventory.drop(columns=["pattern"]))

## Cross-checks

What must be consistent for the pipeline to work:
- **periods**: ERA5 must cover 1981-2015 (training climate), EC-Earth3 historical 1981-2014 (reference of the delta
  method);
- **units**: each notebook converts to mm/day, degC and W m-2 itself, but the units of each source must be known;
- **longitude convention** (0-360 or -180-180) and **calendar** of ERA5 vs EC-Earth3;
- **rasters** (HWSD, slope, GAEZ v5) in geographic coordinates (EPSG:4326);
- one single HWSD raster file must be selected (the notebooks read exactly one).

In [ ]:
ok = inventory[inventory["status"].fillna("").str.startswith("ok")]
if ok.empty:
    print("No readable dataset yet: fill in the TO_FILL patterns and run again.")
else:
    def period(ds_name):
        sub = ok[(ok["dataset"] == ds_name) & ok["t_start"].notna()]
        return (sub["t_start"].min(), sub["t_end"].max()) if not sub.empty else (None, None)

    for ds_name, (need_start, need_end) in {"era5": ("1981-01-01", "2015-12-31"),
                                            "gcm_historical": ("1981-01-01", "2014-12-31")}.items():
        t0, t1 = period(ds_name)
        if t0 is None:
            print(f"{ds_name:18s} not available yet")
        else:
            covered = t0 <= need_start and t1 >= need_end
            print(f"{ds_name:18s} {t0} -> {t1}   {'covers' if covered else 'DOES NOT cover'} {need_start} -> {need_end}")

    nc = ok[ok["kind"] == "netcdf"]
    if not nc.empty:
        print("\nCalendars found:", sorted(nc["calendar"].dropna().unique()))
        print("Units found per variable:")
        print(nc.groupby("var")["units"].unique().to_string())
        lon360 = nc[nc["status"].str.contains("lon 0-360")]
        if not lon360.empty and len(lon360) < len(nc):
            print("\nWARNING: mixed longitude conventions (0-360 and -180-180):", sorted(lon360["dataset"].unique()))

    hw = ok[(ok["dataset"] == "hwsd_raster")]
    if not hw.empty and hw.iloc[0]["n_files"] > 1:
        print(f"\nWARNING: hwsd_raster matches {hw.iloc[0]['n_files']} files ({hw.iloc[0]['files']}):"
              " narrow glob_bdap to the soil mapping unit raster.")

    problems = ok[ok["status"].str.contains("check")]
    if not problems.empty:
        print("\nRows to check:")
        for _, r in problems.iterrows():
            print(f"  - {r['dataset']} / {r['var'] or '-'} : {r['status']}")

todo = inventory[inventory["status"] != "ok"]
todo = todo[~todo["status"].fillna("").str.startswith(("ok", "skipped"))]
if not todo.empty:
    print("\nNot available yet:")
    for _, r in todo.iterrows():
        print(f"  - {r['dataset']:18s} {r['var'] or '-':7s} {r['status']:15s} (used by {r['used_by']})")

## Write the configuration

`config/config_<profile>.json` holds the resolved patterns and the aliases.
The notebooks read it with `crop_utils` instead of redefining paths:

```python
sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()
ERA5_DIR = cu.data_dir(CFG, "era5")               # folder of a dataset
HWSD_TIF = cu.data_file(CFG, "hwsd_raster")       # the single file of a dataset
files    = cu.data_files(CFG, "isimip_mri_ssp585", "pr")   # all files of one variable
```

In [ ]:
config = {
    "profile": PROFILE,
    "data_root": str(DATA_ROOT),
    "project_root": BASE.as_posix(),
    "aliases": ALIASES,
    "datasets": {
        name: {
            "kind": spec["kind"],
            "used_by": spec["used_by"],
            "vars": spec["vars"],
            "template": resolve_pattern(spec, None, template=True),
            "patterns": {var: resolve_pattern(spec, var) for var in var_keys(spec)},
        }
        for name, spec in DATASETS.items()
    },
    "unit_hints": {
        f"{r.dataset}/{r.var}": r.unit_hint
        for r in inventory.itertuples() if isinstance(r.unit_hint, str) and r.unit_hint
    },
}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print("Configuration written:", CONFIG_PATH)

## Next steps

1. Fill in the `TO_FILL` patterns of the `bdap` profile (explore `/eos/jeodpp/data`, e.g. `ls /eos/jeodpp/data/base`),
   run again, fix what the inventory reports (several HWSD files, units, calendar, longitude).
2. If BDAP stores a dataset differently from the local files (e.g. ERA5 one file per year instead of one file per
   variable for 1981-2015), the reading code of the notebooks that use it (column `used_by`) must be adapted.
3. Then run the pipeline in order: `1_SVM/0_preprocessing_masks` -> `1_preprocessing_BIOvariables` -> `2_soil_suitability*`
   -> `3_SVM_train` -> `4_SVM_predict` -> `5_...validation` (-> `6_...AMOC`, `7_test_adaptable_calendar`).

Every notebook starts with the same lines (project root, `crop_utils`, configuration):

```python
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
```